# Tradutor de PDFs no Google Colab — v8.8.0

Este notebook baixa `tradutor_hibrido.py` do seu GitHub, cria as pastas necessárias, permite enviar vários PDFs de uma vez e traduz tudo usando a GPU do Colab.

Uso normal: no Colab, abra **Ambiente de execução → Alterar tipo de ambiente de execução**, selecione **T4 GPU** em *Acelerador de hardware* e confirme em **Salvar**. Depois execute todas as células em ordem e selecione os PDFs quando a janela de upload aparecer.

O Google Tradutor não é usado. Entretanto, por estar no Colab, os documentos são processados em uma máquina virtual do Google e, se o Drive estiver ativado, os resultados e o modelo serão armazenados nele.

## 1. Instalar as dependências

Execute em cada ambiente novo do Colab. O PyTorch fornecido pelo próprio Colab é mantido.

In [ ]:
%pip install -q -U pymupdf "ctranslate2>=4.8,<5" "transformers>=4.44,<5" "accelerate>=0.33,<2" "safetensors>=0.4.3,<1" "sentencepiece>=0.2,<0.3" "langid>=1.1.6,<2" "huggingface_hub[hf_xet]>=0.34,<1"

## 2. Endereço do módulo no GitHub

O notebook está configurado para baixar o módulo do repositório `ProfAllanIFBA/Tradutor_de_Artigos`. O repositório precisa permanecer público. Para fixar uma versão, troque `main` pelo identificador de um commit.

In [ ]:
REPOSITORIO_GITHUB = "ProfAllanIFBA/Tradutor_de_Artigos"
RAMO_OU_COMMIT = "main"
NOME_MODULO = "tradutor_hibrido.py"

URL_MODULO = (
    f"https://raw.githubusercontent.com/{REPOSITORIO_GITHUB}/"
    f"{RAMO_OU_COMMIT}/{NOME_MODULO}"
)

# True: preserva modelo, resultados, cache e retomada no Google Drive.
USAR_GOOGLE_DRIVE = True

# True: copia o modelo do Drive para o SSD temporário do Colab; o carregamento fica mais rápido.
COPIAR_MODELO_PARA_O_COLAB = True

# True: abre automaticamente a janela para baixar um ZIP ao terminar.
BAIXAR_ZIP_AUTOMATICAMENTE = True

## 3. Preparar o ambiente

Esta célula monta o Drive, cria as pastas, recupera o modelo persistente e baixa o módulo do GitHub. Na primeira execução, o Colab pedirá autorização para acessar o Drive.

In [ ]:
from pathlib import Path
import importlib
import re
import shutil
import subprocess
import sys
import urllib.request

if "SEU_USUARIO" in REPOSITORIO_GITHUB or "SEU_REPOSITORIO" in REPOSITORIO_GITHUB:
    raise ValueError("Preencha REPOSITORIO_GITHUB na célula anterior.")

RAIZ_LOCAL = Path("/content/tradutor_pdf")
PASTA_PDFS = RAIZ_LOCAL / "pdf"
PASTA_MODELOS_LOCAL = RAIZ_LOCAL / "modelos_traducao"
PASTA_MODULO = RAIZ_LOCAL / "codigo"
for pasta in (PASTA_PDFS, PASTA_MODELOS_LOCAL, PASTA_MODULO):
    pasta.mkdir(parents=True, exist_ok=True)

if USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ_DRIVE = Path("/content/drive/MyDrive/Tradutor_PDF")
    PASTA_MODELOS_DRIVE = RAIZ_DRIVE / "modelos_traducao"
    PASTA_SAIDA = RAIZ_DRIVE / "resultados"
    PASTA_MODELOS_DRIVE.mkdir(parents=True, exist_ok=True)
    PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    if COPIAR_MODELO_PARA_O_COLAB and any(PASTA_MODELOS_DRIVE.rglob("model.bin")):
        print("Copiando o modelo persistente do Drive para o Colab...")
        shutil.copytree(PASTA_MODELOS_DRIVE, PASTA_MODELOS_LOCAL, dirs_exist_ok=True)
else:
    PASTA_MODELOS_DRIVE = None
    PASTA_SAIDA = PASTA_PDFS / "traduzidos"
    PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

destino_modulo = PASTA_MODULO / NOME_MODULO
print("Baixando o módulo:", URL_MODULO)
with urllib.request.urlopen(URL_MODULO, timeout=60) as resposta:
    codigo = resposta.read()
if len(codigo) < 20_000:
    raise RuntimeError("O arquivo recebido parece incompleto. Confira a URL do GitHub.")
destino_modulo.write_bytes(codigo)

sys.path.insert(0, str(PASTA_MODULO)) if str(PASTA_MODULO) not in sys.path else None
sys.modules.pop("tradutor_hibrido", None)
import tradutor_hibrido
tradutor = importlib.reload(tradutor_hibrido)

def versao_numerica(texto):
    return tuple(int(n) for n in re.findall(r"\d+", str(texto)))

if versao_numerica(tradutor.__version__) < (8, 8, 0):
    raise RuntimeError(f"O GitHub forneceu a versão antiga {tradutor.__version__}.")

import torch
import ctranslate2
if not torch.cuda.is_available() or ctranslate2.get_cuda_device_count() < 1:
    raise RuntimeError("GPU CUDA não disponível. Ative uma GPU nas configurações do ambiente de execução.")

print("Tradutor:", tradutor.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Precisões CUDA:", ctranslate2.get_supported_compute_types("cuda"))
print("Entrada temporária:", PASTA_PDFS)
print("Resultados:", PASTA_SAIDA)

## 4. Escolher o modelo e os ajustes

`nllb_1_3b` oferece a melhor qualidade geral. Se o Colab ficar sem memória durante a preparação, use `nllb_600m`. O modelo é preparado apenas uma vez e depois preservado no Drive.

In [ ]:
MODELO = "nllb_1_3b"  # melhor qualidade; alternativa mais leve: "nllb_600m"
IDIOMAS = ["fr", "en", "it", "es", "pt"]
IDIOMA_DESTINO = "pt"
VARIANTE_PORTUGUES = "pt-BR"
PERFIL_TERMINOLOGICO = "didatica_matematica"  # use None para textos de outras áreas
GLOSSARIO_PERSONALIZADO = {
    # "forma produzida pelo modelo": "forma que você prefere",
}

PROTEGER_ELEMENTOS = True
TRADUZIR_REFERENCIAS = False
UNIR_BLOCOS_CONTIGUOS = True
UNIR_ENTRE_PAGINAS = True
GERAR_AUDITORIA = True
DISPOSITIVO = "cuda"
MAX_CARACTERES_TRECHO = 1600
SOBRESCREVER = False
IGNORAR_JA_TRADUZIDOS = True

AJUSTES = tradutor.MODELOS_LOCAIS[MODELO]
PRECISAO = AJUSTES["precisao_recomendada"]
QUANTIZACAO = AJUSTES["precisao_recomendada"]
FEIXE = None
LOTE_MODELO = None

print(AJUSTES["titulo"], "|", AJUSTES["qualidade"], "|", AJUSTES["velocidade"])

## 5. Preparar o modelo

Na primeira vez, o download e a conversão podem demorar. Ao terminar, a versão convertida é copiada para o Drive. Nas sessões seguintes ela apenas será recuperada.

In [ ]:
caminho_modelo = Path(tradutor.preparar_modelo_local(
    MODELO,
    IDIOMAS,
    IDIOMA_DESTINO,
    pasta_modelos=PASTA_MODELOS_LOCAL,
    quantizacao=QUANTIZACAO,
))

if USAR_GOOGLE_DRIVE:
    relativo = caminho_modelo.relative_to(PASTA_MODELOS_LOCAL)
    destino_persistente = PASTA_MODELOS_DRIVE / relativo
    if not (destino_persistente / "model" / "model.bin").is_file():
        print("Guardando o modelo convertido no Drive. Isso ocorre uma única vez...")
        shutil.copytree(caminho_modelo, destino_persistente, dirs_exist_ok=True)

print("Modelo pronto:", caminho_modelo)

## 6. Enviar os PDFs

Ao executar, aparecerá a janela de seleção. É possível escolher vários PDFs ao mesmo tempo. Os arquivos da seleção anterior são removidos apenas da pasta temporária de entrada; resultados e cache permanecem preservados.

In [ ]:
from google.colab import files

for antigo in PASTA_PDFS.glob("*.pdf"):
    antigo.unlink()

enviados = files.upload()
pdfs_salvos = []
for nome, conteudo in enviados.items():
    nome_seguro = Path(nome).name
    if Path(nome_seguro).suffix.lower() != ".pdf":
        print("Ignorado por não ser PDF:", nome_seguro)
        continue
    destino = PASTA_PDFS / nome_seguro
    destino.write_bytes(conteudo)
    pdfs_salvos.append(destino)

if not pdfs_salvos:
    raise RuntimeError("Nenhum PDF foi enviado. Execute esta célula novamente.")

print(f"{len(pdfs_salvos)} PDF(s) pronto(s) para tradução:")
for pdf in pdfs_salvos:
    print(" -", pdf.name)

## 7. Traduzir todos os PDFs

O lote continua mesmo se um trecho ou um arquivo apresentar erro. Com o Drive ativado, relatórios, pendências e progresso ficam em `Meu Drive/Tradutor_PDF/resultados`.

In [ ]:
resumo = tradutor.traduzir_pasta(
    PASTA_PDFS,
    pasta_saida=PASTA_SAIDA,
    idioma_origem=IDIOMAS,
    idioma_destino=IDIOMA_DESTINO,
    variante_portugues=VARIANTE_PORTUGUES,
    perfil_terminologico=PERFIL_TERMINOLOGICO,
    glossario_personalizado=GLOSSARIO_PERSONALIZADO,
    proteger_elementos=PROTEGER_ELEMENTOS,
    traduzir_referencias=TRADUZIR_REFERENCIAS,
    unir_blocos_contiguos=UNIR_BLOCOS_CONTIGUOS,
    unir_entre_paginas=UNIR_ENTRE_PAGINAS,
    gerar_auditoria=GERAR_AUDITORIA,
    motor_traducao="local",
    modelo_local=MODELO,
    pasta_modelos=PASTA_MODELOS_LOCAL,
    quantizacao_modelo=QUANTIZACAO,
    precisao_local=PRECISAO,
    dispositivo_local=DISPOSITIVO,
    feixe=FEIXE,
    lote_modelo=LOTE_MODELO,
    max_caracteres_trecho=MAX_CARACTERES_TRECHO,
    ao_erro_traducao="continuar",
    motor_texto="nativo",
    salvar_a_cada=1,
    sobrescrever=SOBRESCREVER,
    ignorar_ja_traduzidos=IGNORAR_JA_TRADUZIDOS,
)

print("Estado:", resumo["estado"])
print("Totais:", resumo["totais"])
print("Saída:", resumo["pasta_saida"])

## 8. Baixar os resultados

Cria um ZIP com os PDFs traduzidos, relatórios, pendências e auditorias. Se o navegador impedir o download automático, o arquivo continuará disponível no painel de arquivos do Colab e, com o Drive ativado, os resultados permanecerão no Drive.

In [ ]:
arquivo_zip = Path(shutil.make_archive(
    "/content/traducoes_pdf",
    "zip",
    root_dir=PASTA_SAIDA,
))
print("ZIP criado:", arquivo_zip)

if BAIXAR_ZIP_AUTOMATICAMENTE:
    files.download(str(arquivo_zip))

## Uso nas próximas vezes

Abra o notebook pelo GitHub no Colab e execute as células em ordem. O módulo será atualizado pelo GitHub automaticamente. O modelo já convertido será recuperado do Drive; basta selecionar novos PDFs na etapa 6.

Se uma sessão cair, execute novamente as células. Como a saída e os arquivos de controle estão no Drive, o tradutor poderá aproveitar o que já foi concluído.